Example of using GeoJSON data sources

# Ecosystems Map

In [ ]:
import logging
logging.getLogger('rle_python_gee.aoo').setLevel(logging.INFO)
logging.basicConfig(level=logging.INFO)

In [ ]:
from rle_python_gee.ecosystems import Ecosystems
from rle_python_gee.aoo import make_aoo

In [ ]:
TEST_EXPORTS = True

In [ ]:
# ecosystems = Ecosystems.from_file('../tests/test_data/null_island.geojson', ecosystem_column='ECO_CODE')
ecosystems = Ecosystems.from_file(
    '/Users/tylere/Documents/GitHub/RLE-Assessment/rle-python-gee/tests/test_data/colombia_ecosystems.shp',
    ecosystem_column='ECO_CODE'
)
ecosystems.load()
ecosystems

In [ ]:
ecosystems.head()

In [ ]:
ecosystems.unique_ecosystems()

In [ ]:
ecosystems.to_map()

In [ ]:
if TEST_EXPORTS:
    ecosystems.to_parquet('/tmp/ecosystems.parquet')

In [ ]:
if TEST_EXPORTS:    
    import ee
    ee.Initialize(project='goog-rle-assessments')
    task_id = ecosystems.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/ecosystems',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')

# AOO Grid

In [ ]:
aoo_grid = make_aoo(ecosystems).compute()
aoo_grid

In [ ]:
aoo_grid.grid_cells.head()

In [ ]:
aoo_grid.to_map()

In [ ]:
aoo_grid.filter_by_ecosystem('F1_1_2', threshold=0.00).to_map()

In [ ]:
if TEST_EXPORTS:
    aoo_grid.to_parquet('/tmp/aoo_grid.parquet')

In [ ]:
if TEST_EXPORTS:
    task_id = aoo_grid.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/aoo_grid',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')

# AOO Grid Polygons

In [ ]:
aoo_grid_polygons = aoo_grid.to_polygons().compute()
aoo_grid_polygons

In [ ]:
aoo_grid_polygons.to_map()

In [ ]:
if TEST_EXPORTS:
    # Write the grid polygons to a parquet file
    aoo_grid_polygons.to_parquet('/tmp/aoo_grid_polygons.parquet')

In [ ]:
if TEST_EXPORTS:
    task_id = aoo_grid_polygons.to_ee_feature_collection(
        'projects/goog-rle-assessments/assets/temp/aoo_grid_polygons',
        gcs_bucket='rle_test_bucket_1'
    )
    print(f'{task_id=}')

# Layered Map

In [ ]:
from lonboard import Map

Map(layers=ecosystems.to_layer() + aoo_grid_polygons.to_layer())